<a href="https://colab.research.google.com/github/ks-chauhan/Brain-Tumor-Detection/blob/main/Transfer_Learning_with_ResNet_50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Dataset Download

import kagglehub
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Path to dataset files: /kaggle/input/brain-tumor-mri-dataset


In [2]:
# Import Libraries

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import ImageFolder
from torchvision import transforms, models
from torch.utils.data import DataLoader

In [3]:
# Image Preprocessing

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [4]:
# Load Dataset

train_dataset = ImageFolder(
    "/kaggle/input/brain-tumor-mri-dataset/Training",
    transform=transform
)
test_dataset = ImageFolder(
    "/kaggle/input/brain-tumor-mri-dataset/Testing",
    transform=transform
)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

classes = train_dataset.classes
print(classes)

['glioma', 'meningioma', 'notumor', 'pituitary']


In [5]:
# ResNet 50 Loading

model = models.resnet50(weights="IMAGENET1K_V1")
model.fc = nn.Linear(model.fc.in_features, 4)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 158MB/s]


In [6]:
# Move Model to GPU

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [7]:
# Loss & Optimizer

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [8]:
# Training Loop

epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {running_loss}")

Epoch 1 Loss: 40.91114036086947
Epoch 2 Loss: 12.029952518176287
Epoch 3 Loss: 9.68219197052531
Epoch 4 Loss: 3.2168306476669386
Epoch 5 Loss: 3.6944785358500667


In [9]:
# Accuracy Evaluation

correct = 0
total = 0
model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs,1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy:",100*correct/total)

Accuracy: 94.25


In [10]:
# Save Model

torch.save(model.state_dict(), "brain_tumor_resnet50.pth")